# D2.2 · Generating detection rules from an incident

**Function D — The Agentic SOC → The Agentic SOC — Detect**

Builds on **[D2.1 · Agent-assisted detection engineering](https://spbreed.github.io/cyber-commons/lessons/D2.1.html)**.

| | |
|---|---|
| Tools used | Sigma |

## What this lesson is

**What it covers.** Generating a detection rule from a reconstructed incident, and measuring it against benign traffic before it ships.

**Why a security engineer needs it.** An incident is the richest source of a good rule and the easiest source of a bad one, because every candidate you write from it catches it. The property that decides deployability is the false-positive rate on traffic that is not the incident — which means a benign corpus containing the hard cases, not unrelated noise.

## 1 · The hook

You have just reconstructed an incident and the rule almost writes itself. That is the problem: every rule you could write catches the incident, because you wrote it from the incident. What decides whether it ships is the traffic it fires on when nothing is wrong.

> **At CyberTravels.** The incident is CyberTravels': the Workflow Agent issued a refund against a booking nobody asked it to touch. The benign corpus is the hard one on purpose — CyberTravels processes eighteen legitimate refunds in the same window, and a rule that cannot tell them apart is a rule that alerts on the business.

## 2 · The framework

```
   incident trace
        |
        v
   +----------------------+     every candidate catches the incident,
   | candidate rules      |     so that cannot be the selection test
   |   any refund         |
   |   refund w/o approval|
   |   refund on BK-772   |
   +----------+-----------+
              |
              v
       benign corpus  (must contain LEGITIMATE refunds)
              |
      +-------+--------+---------------+
      v                v               v
   21% FP           0% FP           0% FP
   buries the       generalises     matches this
   queue            AND quiet       incident only
   REJECT           SHIP            REJECT
```

The fastest source of a good detection is an incident you have just had. The
trap is that **every candidate rule catches the incident** — that is how it was
generated — so catching it cannot be the property you select on.

What separates a rule worth deploying is what it does to traffic that is not the
incident. That means a benign corpus containing the hard cases: for a refund
rule, legitimate refunds. A corpus of unrelated traffic proves nothing, because
the naive rule looks clean against it.

## 3 · Write the bad candidates on purpose

Two rules are worth writing precisely because they will be rejected.

The **naive generalisation** takes the tool that appeared in the incident:
`any refund`. The **over-fitted** one takes the identifier: `refund on BK-772`.
Both catch the incident. One buries the queue and the other is worthless
tomorrow, and seeing them scored beside the good rule is what makes the good
rule a choice rather than an assumption.

## 4 · Generate, then measure before shipping

Three candidates, one benign corpus, one number each. A rule with no measured false-positive rate is a guess with syntax.

### The skill — [`skills/detection/detection-rule-synthesis/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/detection/detection-rule-synthesis/SKILL.md)

```yaml
name: detection-rule-synthesis
description: >-
  Generate a detection rule from an incident trace and score it against benign
  traffic before it ships. Use after an incident is reconstructed, when turning
  a hunt finding into a standing detection, or when a proposed rule has no
  measured false-positive rate.
allowed-tools: Read, Grep, Glob
```

# The incident tells you what to write; the benign corpus tells you whether to ship it

The fastest source of a good detection is an incident you have just had. The
trap is that **every candidate rule catches the incident** — that is how it was
generated — so catching it cannot be the property you select on.

What separates a rule worth deploying is what it does to traffic that is *not*
the incident.

## When to use this

Immediately after reconstruction, while the trace is still loaded, and again
whenever a hunt hypothesis is promoted. Do not use it to write coverage from
scratch: a rule generated from no incident has nothing to generalise from.

## Step-by-step

**1 — Take the reconstructed trace, not the alert.** The alert is one event;
the rule needs the sequence around it.

**2 — Write more than one candidate, deliberately including a bad one.** The
naive generalisation ("the tool that appeared") and the over-fitted one ("this
booking id") are both worth writing, because seeing them scored is the lesson.

**3 — Assemble a benign corpus that contains the hard cases.** Legitimate
refunds, not just unrelated traffic. A corpus with no near-misses proves
nothing.

**4 — Score every candidate on the benign corpus and record the rate.** Not a
verdict, a number.

**5 — Ship the one that generalises and stays quiet, and say why the others
were rejected.** The rejected candidates are the evidence that the shipped one
was chosen rather than assumed.

## Example

**Input** — a three-step incident and an 84-run benign corpus, in
[`scripts/detection_rule_synthesis.py`](scripts/detection_rule_synthesis.py).

**Output** — a real run:

```
candidate rule                          fires on   FP  FP rate  verdict
any refund                                  True   18     21%  REJECT — buries the queue
refund with no approval in the run          True    0      0%  ship
refund on BK-772                            True    0      0%  REJECT — matches this incident only
```

All three catch the incident. Only one is a detection.

## Output contract

```json
{
  "benign_runs": 0,
  "candidates": [{"rule": "str", "catches_incident": true,
                  "false_positives": 0, "fp_rate": 0.0,
                  "verdict": "ship|REJECT"}]
}
```

## Common edge cases

- **The over-fitted rule scores perfectly.** Zero false positives and zero
  future value. Test for a literal identifier in the rule body.
- **The benign corpus is too easy.** If it contains no legitimate refunds, the
  naive rule looks shippable.
- **The incident is the only positive.** One true positive cannot establish
  recall; the rule's recall is unknown until it runs.

## Failure modes

- **Selecting on "does it catch the incident".** Every candidate does.
- **Shipping without a false-positive number.** That is a guess with syntax.
- **Never writing the bad candidates.** Then nothing shows the good one is good.

In [ ]:
# The code is not in this notebook. It is this file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/detection/detection-rule-synthesis/scripts/detection_rule_synthesis.py
SCRIPT = "skills/detection/detection-rule-synthesis/scripts/detection_rule_synthesis.py"
REPO = "https://github.com/spbreed/cyber-commons"
BRANCH = "claude/vulnbench-setup-scheduling-81aqov"

import glob, os, subprocess, sys

CLONE = "/kaggle/working/cyber-commons"
_root = next((r for r in (".", "..", "../..", CLONE)
              if os.path.isfile(os.path.join(r, SCRIPT))), None)

if _root is None:
    # --filter=blob:none --sparse fetches the tree without the history or the
    # notebooks; sparse-checkout then materialises only the two directories a
    # lesson needs: the procedures, and the repository they are run against.
    _c = subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                         "--sparse", "--branch", BRANCH, REPO, CLONE],
                        capture_output=True, text=True)
    if _c.returncode:
        raise SystemExit(
            "could not fetch the skills: " + _c.stderr.strip()[-300:] +
            "\nOn Kaggle this needs Internet on in the notebook settings, which "
            "needs a phone-verified account. Without one, attach the dataset "
            "cybercommons/cyber-commons-skills instead — it holds the same tree.")
    # `skills` is the procedures; `cybertravels` is the sample repository they
    # scan; `curriculum` and `site/data` hold the framework mapping and the
    # session list that the reference-lookup skill reads. Miss any of them and
    # the skill clones successfully and then fails on a path that is not there,
    # which is how A0.2 failed its first Kaggle run.
    subprocess.run(["git", "-C", CLONE, "sparse-checkout", "set",
                    "skills", "cybertravels", "curriculum", "site/data"],
                   capture_output=True, text=True)
    _root = CLONE

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## What you just proved

All three candidates catch the incident; only the sequence rule ships. The naive rule fires on 18 legitimate refunds — a 21% false-positive rate on a business that runs on refunds.

## Your turn

Add a benign run that the shipped rule fires on. If you cannot construct one, your corpus is too easy.

---

**Next → [D2.3 · Detection engineering *for* agents](https://spbreed.github.io/cyber-commons/lessons/D2.3.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/D2.2.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/D2.2.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*